
# MS-ILA — Day 05 / TV1-B Real-data Integration Check (Colab T4)

Notebook này **không phải training/ablation chính thức**. Nó chỉ kiểm tra framework trên vài ảnh `MVTec AD 2 / train` thật theo chuỗi:

`real image → Local/Context → frozen DINOv3 → Adapter → Alignment (R2) → Projection → FeatureSelector → MeanFusion → Decoder → loss → backward → optimizer.step()`

### Hai gate phải PASS

1. **R0/R1/R2 integration**
   - loss finite, logits/anomaly map finite;
   - DINOv3 frozen và không nhận gradient;
   - module trainable có gradient;
   - `optimizer.step()` thực sự làm thay đổi weight của các branch đang dùng;
   - anomaly map `.png` được sinh.

2. **R2 source usage**
   - đủ 6 nguồn `L4,L8,L12,C4,C8,C12` sau alignment/projection;
   - cả 6 tensor đều có gradient finite và non-zero;
   - không source nào bị silently ignored.

### Về `r,d` chưa chốt

Notebook dùng **smoke-only adapter** mặc định `r=32, d=128` để chạy nhanh trên Colab T4. Đây **không phải Day-04 winner** và tuyệt đối không được dùng để kết luận R0/R1/R2 tốt/xấu.

Sau này chỉ cần sửa hai biến `SMOKE_ADAPTER_R`, `SMOKE_ADAPTER_D` thành winner Day 04 rồi chạy lại. Mọi candidate vẫn dùng cùng một `(r,d)`.

> `gamma_init=0` làm Adapter bắt đầu như identity. Vì vậy notebook chạy mặc định **3 optimizer steps**: step đầu chủ yếu mở residual gate `gamma`; từ step sau các weight bên trong adapter mới có điều kiện cập nhật rõ ràng.


In [ ]:

# ============================================================
# 0. COLAB + GOOGLE DRIVE
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, sys, json, math, random, copy, hashlib, time
import numpy as np
import torch
import yaml
from PIL import Image
import matplotlib.pyplot as plt

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "Notebook này được thiết kế cho Colab GPU. "
        "Chọn Runtime -> Change runtime type -> T4 GPU rồi chạy lại."
    )

DEVICE = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:

# ============================================================
# 1. PROJECT PATHS — KHỚP CẤU TRÚC REPO HIỆN TẠI
# ============================================================
PROJECT_ROOT = Path("/content/msila")
DEFAULT_CONFIG_PATH = PROJECT_ROOT / "configs" / "default.yaml"
DAY05_CONFIG_PATH = PROJECT_ROOT / "configs" / "day05_representation.yaml"

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        "Không thấy /content/msila. Hãy clone/copy repo MS-ILA vào đúng path này."
    )
if not DEFAULT_CONFIG_PATH.is_file():
    raise FileNotFoundError(DEFAULT_CONFIG_PATH)
if not DAY05_CONFIG_PATH.is_file():
    raise FileNotFoundError(
        f"{DAY05_CONFIG_PATH}\n"
        "Cần file Day-05 framework từ task trước."
    )

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

with open(DEFAULT_CONFIG_PATH, "r", encoding="utf-8") as f:
    DEFAULT_CFG = yaml.safe_load(f)

with open(DAY05_CONFIG_PATH, "r", encoding="utf-8") as f:
    DAY05_CFG = yaml.safe_load(f)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Default backbone:", DEFAULT_CFG["backbone"])
print("Day05 candidates:", DAY05_CFG["experiment"]["candidate_ids"])


In [ ]:

# ============================================================
# 2. SMOKE-ONLY CONFIG
#    CHỈ SỬA CELL NÀY KHI MUỐN ĐỔI r,d / category / số step
# ============================================================

CATEGORY = "fabric"

# TEMPORARY ONLY — KHÔNG PHẢI DAY-04 WINNER
SMOKE_ADAPTER_R = 32
SMOKE_ADAPTER_D = 128

# gamma_init=0 => nên >= 2 step; dùng 3 để audit weight update rõ hơn.
STEPS_PER_CANDIDATE = 3
BATCH_SIZE = 1

SMOKE_LR = 1e-3
SMOKE_WEIGHT_DECAY = 0.0

# DINO frozen nên có thể cache feature RAM một lần rồi tái dùng cho R0/R1/R2.
# Điều này tiết kiệm T4 và KHÔNG thay đổi representation giữa candidates.
REUSE_FROZEN_DINO_FEATURES_IN_RAM = True

SEED = int(DEFAULT_CFG["training"]["seed"])
CANDIDATES = ("R0", "R1", "R2")

# Các giá trị kiến trúc KHÔNG tune trong smoke.
LOCAL_SIZE = int(DEFAULT_CFG["input"]["tile_size"])
CONTEXT_SIZE = int(DEFAULT_CFG["input"]["context_size"])
BLOCKS = tuple(int(x) for x in DEFAULT_CFG["backbone"]["feature_blocks"])
MODEL_NAME = str(DEFAULT_CFG["backbone"]["name"])
FUSION_DIM = int(DAY05_CFG["locked"]["projection"]["fusion_dim"])
ADAPTER_KERNEL = int(DAY05_CFG["locked"]["adapter"]["kernel_size"])
GAMMA_INIT = float(DAY05_CFG["locked"]["adapter"]["gamma_init"])

assert BATCH_SIZE == 1, "Notebook smoke hiện khóa BATCH_SIZE=1 để geometry audit đơn giản/rõ."
assert STEPS_PER_CANDIDATE >= 2, "gamma_init=0 => cần ít nhất 2 optimizer steps."
assert CANDIDATES == tuple(DAY05_CFG["experiment"]["candidate_ids"])
assert BLOCKS == (4, 8, 12)
assert bool(DEFAULT_CFG["backbone"]["frozen"]) is True
assert DAY05_CFG["locked"]["fusion"]["type"] == "mean"

SMOKE_CFG = {
    "purpose": "day05_tv1b_integration_smoke_only",
    "scientific_result": False,
    "category": CATEGORY,
    "candidates": list(CANDIDATES),
    "steps_per_candidate": STEPS_PER_CANDIDATE,
    "batch_size": BATCH_SIZE,
    "adapter": {
        "bottleneck_dim": SMOKE_ADAPTER_R,
        "projection_dim": SMOKE_ADAPTER_D,
        "source": "temporary_smoke_not_day04_winner",
        "kernel_size": ADAPTER_KERNEL,
        "gamma_init": GAMMA_INIT,
    },
    "projection": {"fusion_dim": FUSION_DIM},
    "fusion": {"type": "mean", "trainable": False},
    "optimizer": {
        "name": "AdamW",
        "learning_rate": SMOKE_LR,
        "weight_decay": SMOKE_WEIGHT_DECAY,
    },
    "seed": SEED,
}

SMOKE_CONFIG_PATH = PROJECT_ROOT / "configs" / "day05_integration_smoke.yaml"
SMOKE_CONFIG_PATH.write_text(
    yaml.safe_dump(SMOKE_CFG, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)

print(yaml.safe_dump(SMOKE_CFG, sort_keys=False, allow_unicode=True))
print("Saved:", SMOKE_CONFIG_PATH)
print("\nIMPORTANT: smoke r,d chỉ để test framework, KHÔNG dùng làm kết quả ablation.")


In [ ]:

# ============================================================
# 3. IMPORT ĐÚNG MODULE THEO PROJECT
# ============================================================
from src.data.multiview_transform import (
    MultiViewConfig,
    NestedMultiViewTransform,
    IMAGENET_MEAN,
    IMAGENET_STD,
)
from src.geometry.view_meta import build_view_meta_from_transform_meta

from src.models.dinov3_extractor import build_online_extractor
from src.models.residual_adapter import ResidualAdapter2d
from src.models.context_alignment import ContextToLocalAligner
from src.models.feature_projection import SixFeatureProjection
from src.models.feature_selector import FeatureSelector
from src.models.mean_fusion import MeanFusion
from src.models.basic_decoder import BasicDecoder

from src.losses.anomaly_loss import AnomalySegmentationLoss

# API gates: nếu repo chưa sync module Day-05 mới thì fail rõ ngay.
if not hasattr(SixFeatureProjection, "project_sources"):
    raise RuntimeError(
        "src/models/feature_projection.py chưa phải bản Day-05: "
        "thiếu SixFeatureProjection.project_sources()."
    )

try:
    _ = MeanFusion(validate=True, return_weights=True)
except TypeError as exc:
    raise RuntimeError(
        "src/models/mean_fusion.py chưa phải bản Day-05 flexible 1/3/6 sources."
    ) from exc

print("Project imports: PASS")


In [ ]:

# ============================================================
# 4. RESOLVE REAL MVTec AD 2 DATA + OUTPUT PATH
# ============================================================
DATA_ROOT = Path(DEFAULT_CFG["data"]["mvtec_ad2"]).expanduser()
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(
        f"Không thấy data.mvtec_ad2 từ configs/default.yaml:\n{DATA_ROOT}"
    )

TEAM_ROOT = DATA_ROOT.parent
OUTPUT_ROOT = TEAM_ROOT / "outputs" / "day05" / "integration_check"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}

def resolve_category_train_dir(data_root: Path, category: str) -> Path:
    # Fast/direct layouts first.
    direct = [
        data_root / category / "train",
        data_root / "mvtec_ad_2" / category / "train",
        data_root / "mvtec_ad2" / category / "train",
        data_root / "MVTec AD 2" / category / "train",
    ]
    for p in direct:
        if p.is_dir():
            return p

    # Fallback: search only for a directory named <category>/train.
    matches = []
    for train_dir in data_root.rglob("train"):
        if train_dir.is_dir() and train_dir.parent.name.lower() == category.lower():
            matches.append(train_dir)

    if len(matches) == 1:
        return matches[0]
    if not matches:
        raise FileNotFoundError(
            f"Không tìm thấy {category}/train dưới {data_root}. "
            "Kiểm tra cấu trúc MVTec AD 2 trên Drive."
        )
    raise RuntimeError(
        "Tìm thấy nhiều train directories cho category. "
        "Hãy chỉnh DATA_ROOT/CATEGORY cho duy nhất:\n" +
        "\n".join(str(p) for p in matches)
    )

TRAIN_DIR = resolve_category_train_dir(DATA_ROOT, CATEGORY)
REAL_TRAIN_IMAGES = sorted(
    p for p in TRAIN_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
)

if len(REAL_TRAIN_IMAGES) < STEPS_PER_CANDIDATE:
    raise RuntimeError(
        f"Cần ít nhất {STEPS_PER_CANDIDATE} ảnh train thật, "
        f"chỉ thấy {len(REAL_TRAIN_IMAGES)} ở {TRAIN_DIR}"
    )

print("DATA_ROOT :", DATA_ROOT)
print("TRAIN_DIR :", TRAIN_DIR)
print("Images    :", len(REAL_TRAIN_IMAGES))
print("OUTPUT    :", OUTPUT_ROOT)
print("First 3   :", [str(p.name) for p in REAL_TRAIN_IMAGES[:3]])


In [ ]:

# ============================================================
# 5. RESOLVE DINOv3 REPO + CHECKPOINT
# ============================================================
# Có thể override thủ công 2 biến này nếu auto-detect không đúng:
DINOV3_REPO = Path(os.getenv("DINOV3_REPO", "/content/dinov3_repo"))
DINOV3_WEIGHTS_OVERRIDE = os.getenv("DINOV3_WEIGHTS", "").strip()

if not (DINOV3_REPO / "hubconf.py").is_file():
    print("Không thấy DINOv3 repo tại", DINOV3_REPO)
    print("Đang clone official facebookresearch/dinov3 ...")
    !rm -rf /content/dinov3_repo
    !git clone -q https://github.com/facebookresearch/dinov3.git /content/dinov3_repo

if not (DINOV3_REPO / "hubconf.py").is_file():
    raise FileNotFoundError(f"DINOv3 repo không hợp lệ: {DINOV3_REPO}")

def resolve_weights() -> Path:
    if DINOV3_WEIGHTS_OVERRIDE:
        p = Path(DINOV3_WEIGHTS_OVERRIDE).expanduser()
        if not p.is_file():
            raise FileNotFoundError(f"DINOV3_WEIGHTS không tồn tại: {p}")
        return p

    roots = [
        TEAM_ROOT / "weights",
        PROJECT_ROOT / "weights",
        Path("/content/checkpoints"),
    ]
    patterns = [
        f"*{MODEL_NAME}*.pth",
        "*dinov3*vitb16*.pth",
        "*vitb16*.pth",
    ]
    found = set()
    for root in roots:
        if not root.is_dir():
            continue
        for pattern in patterns:
            found.update(p.resolve() for p in root.rglob(pattern) if p.is_file())

    found = sorted(found)
    if len(found) == 1:
        return found[0]
    if not found:
        raise FileNotFoundError(
            "Không tự tìm thấy DINOv3 ViT-B/16 checkpoint. "
            "Hãy đặt checkpoint vào TEAM_ROOT/weights hoặc chạy trước cell:\n"
            "os.environ['DINOV3_WEIGHTS']='/content/drive/.../checkpoint.pth'"
        )

    print("Có nhiều checkpoint phù hợp:")
    for p in found:
        print(" -", p)
    raise RuntimeError(
        "Có nhiều checkpoint. Set os.environ['DINOV3_WEIGHTS'] "
        "đúng file rồi chạy lại cell này."
    )

DINOV3_WEIGHTS = resolve_weights()

print("DINO repo   :", DINOV3_REPO)
print("DINO weights:", DINOV3_WEIGHTS)
print("Model       :", MODEL_NAME)


In [ ]:

# ============================================================
# 6. REPRODUCIBILITY + REAL LOCAL/CONTEXT BATCHES
# ============================================================
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

# Center crop cho integration check: deterministic và geometry dễ audit.
# Đây KHÔNG thay thế sampling policy của training chính thức.
mv = NestedMultiViewTransform(
    MultiViewConfig(
        local_size=LOCAL_SIZE,
        context_size=CONTEXT_SIZE,
        input_size=LOCAL_SIZE,
        sampling="center",
        normalize=True,
        validate_output=True,
    )
)

def prepare_one_real_sample(path: Path):
    image = Image.open(path).convert("RGB")
    view = mv(image)
    geom = build_view_meta_from_transform_meta(
        view["meta"],
        local_input_hw=(LOCAL_SIZE, LOCAL_SIZE),
        context_input_hw=(LOCAL_SIZE, LOCAL_SIZE),
        validate=True,
    )
    return {
        "path": path,
        "x_local": view["x_local"].unsqueeze(0),
        "x_context": view["x_context"].unsqueeze(0),
        "geometry": geom,
    }

REAL_BATCHES = []
for p in REAL_TRAIN_IMAGES:
    try:
        sample = prepare_one_real_sample(p)
    except ValueError as exc:
        # multiview_transform chủ động reject ảnh nhỏ hơn Context FOV.
        print("[SKIP]", p.name, "->", exc)
        continue
    REAL_BATCHES.append(sample)
    if len(REAL_BATCHES) == STEPS_PER_CANDIDATE:
        break

if len(REAL_BATCHES) < STEPS_PER_CANDIDATE:
    raise RuntimeError(
        "Không đủ ảnh train hợp lệ (>= Context 768) cho integration check."
    )

for i, b in enumerate(REAL_BATCHES):
    print(
        f"batch {i}:",
        b["path"].name,
        tuple(b["x_local"].shape),
        tuple(b["x_context"].shape),
    )


In [ ]:

# ============================================================
# 7. LOAD REAL FROZEN DINOv3
# ============================================================
extractor = build_online_extractor(
    repo_dir=DINOV3_REPO,
    weights=DINOV3_WEIGHTS,
    device=DEVICE,
    model_name=MODEL_NAME,
    blocks=BLOCKS,
    norm=True,
    check_finite=True,
)

BACKBONE_PARAMS = list(extractor.backbone.parameters())
assert BACKBONE_PARAMS
assert extractor.backbone_is_frozen()
assert all(not p.requires_grad for p in BACKBONE_PARAMS)
assert extractor.backbone.training is False

print("Frozen DINOv3: PASS")
print("out_channels :", extractor.out_channels)
print("patch_size   :", extractor.patch_size)

# Cache frozen features in RAM/CPU once. This is valid for this smoke test
# because DINO is frozen and identical for all R0/R1/R2.
FROZEN_BATCHES = []
if REUSE_FROZEN_DINO_FEATURES_IN_RAM:
    print("\nExtracting frozen features once from REAL train images...")
    with torch.no_grad():
        for i, b in enumerate(REAL_BATCHES):
            x_local = b["x_local"].to(DEVICE, non_blocking=True)
            x_context = b["x_context"].to(DEVICE, non_blocking=True)
            online = extractor.extract_local_context(
                x_local,
                x_context,
                strategy="concat",
            )
            assert set(online) == {"L4", "L8", "L12", "C4", "C8", "C12"}
            assert all(torch.isfinite(v).all() for v in online.values())
            assert all(not v.requires_grad for v in online.values())

            FROZEN_BATCHES.append({
                "path": b["path"],
                "geometry": b["geometry"],
                "x_local_cpu": b["x_local"].clone(),
                "online_cpu": {
                    k: v.detach().cpu().contiguous()
                    for k, v in online.items()
                },
            })
            print(i, b["path"].name, {k: tuple(v.shape) for k, v in online.items()})

    torch.cuda.empty_cache()
    print("Frozen feature extraction: PASS")
else:
    print("REUSE_FROZEN_DINO_FEATURES_IN_RAM=False: DINO sẽ chạy lại theo step.")


In [ ]:

# ============================================================
# 8. DAY-05 CANDIDATE MODEL — MỘT CODEBASE, CHỈ ĐỔI SELECTOR
# ============================================================
from torch import nn

class Day05RepresentationSmokeModel(nn.Module):
    def __init__(self, candidate_id: str, in_channels: int):
        super().__init__()
        self.candidate_id = str(candidate_id).upper()
        self.selector = FeatureSelector.from_config(
            DAY05_CFG,
            self.candidate_id,
            validate=True,
        )

        # Cùng 3 block-specific adapters cho mọi candidate.
        # r,d giống nhau tuyệt đối giữa R0/R1/R2.
        self.adapters = nn.ModuleDict({
            f"b{block}": ResidualAdapter2d(
                in_dim=int(in_channels),
                bottleneck_dim=int(SMOKE_ADAPTER_R),
                projection_dim=int(SMOKE_ADAPTER_D),
                kernel_size=int(ADAPTER_KERNEL),
                gamma_init=float(GAMMA_INIT),
                bias=True,
            )
            for block in BLOCKS
        })

        self.aligner = ContextToLocalAligner(
            check_finite=True,
            check_bounds=True,
        )

        self.projection = SixFeatureProjection(
            in_channels=int(in_channels),
            fusion_dim=int(FUSION_DIM),
            blocks=BLOCKS,
            share_across_views=True,
            check_finite=True,
        )

        self.fusion = MeanFusion(
            validate=True,
            return_weights=True,
        )
        self.decoder = BasicDecoder(
            in_channels=int(FUSION_DIM),
        )

    @property
    def active_blocks(self):
        return (12,) if self.candidate_id == "R0" else BLOCKS

    def forward(
        self,
        online_features,
        geometry,
        *,
        output_size=(LOCAL_SIZE, LOCAL_SIZE),
        retain_source_grads=False,
    ):
        local = {
            f"L{block}": self.adapters[f"b{block}"](online_features[f"L{block}"])
            for block in self.active_blocks
        }

        aligned_context = None
        if self.candidate_id == "R2":
            context = {
                f"C{block}": self.adapters[f"b{block}"](online_features[f"C{block}"])
                for block in BLOCKS
            }
            h, w = local["L4"].shape[-2:]
            aligned_context = self.aligner(
                context,
                geometry,
                target_hw=(int(h), int(w)),
            )

        projected = self.projection.project_sources(
            local,
            aligned_context,
            source_keys=self.selector.source_keys,
        )

        if retain_source_grads:
            for tensor in projected.values():
                tensor.retain_grad()

        selected = self.selector(projected)
        fused, weights = self.fusion(selected)
        logits = self.decoder(fused, output_size=output_size)

        return logits, {
            "projected": projected,
            "selected": selected,
            "fused": fused,
            "fusion_weights": weights,
            "aligned_context": aligned_context,
        }


def snapshot_parameters(module: nn.Module):
    return {
        name: p.detach().cpu().clone()
        for name, p in module.named_parameters()
    }


def changed_parameter_names(before, module: nn.Module, atol=0.0):
    current = dict(module.named_parameters())
    changed = []
    for name, old in before.items():
        new = current[name].detach().cpu()
        if atol == 0.0:
            is_changed = not torch.equal(old, new)
        else:
            is_changed = not torch.allclose(old, new, atol=atol, rtol=0.0)
        if is_changed:
            changed.append(name)
    return changed


def grad_stats_for_module(module: nn.Module):
    trainable = [(n, p) for n, p in module.named_parameters() if p.requires_grad]
    with_grad = [(n, p.grad) for n, p in trainable if p.grad is not None]
    return {
        "trainable_tensors": len(trainable),
        "with_grad": len(with_grad),
        "finite": all(bool(torch.isfinite(g).all()) for _, g in with_grad),
        "nonzero_tensors": sum(
            bool((g != 0).any()) for _, g in with_grad
        ),
        "grad_l1": float(sum(g.detach().abs().sum().item() for _, g in with_grad)),
    }


print("Day05RepresentationSmokeModel: READY")


In [ ]:

# ============================================================
# 9. REAL-DATA INTEGRATION:
#    forward -> loss -> backward -> optimizer.step()
# ============================================================
criterion = AnomalySegmentationLoss(
    bce_weight=float(DAY05_CFG["locked"]["loss"]["bce_weight"]),
    dice_weight=float(DAY05_CFG["locked"]["loss"]["dice_weight"]),
).to(DEVICE)

def denormalize_local(x):
    mean = torch.tensor(IMAGENET_MEAN, dtype=x.dtype).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD, dtype=x.dtype).view(3, 1, 1)
    return (x.cpu() * std + mean).clamp(0, 1)

def get_online_batch(step_idx):
    if REUSE_FROZEN_DINO_FEATURES_IN_RAM:
        rec = FROZEN_BATCHES[step_idx]
        online = {
            k: v.to(DEVICE, non_blocking=True)
            for k, v in rec["online_cpu"].items()
        }
        return rec["path"], rec["geometry"], rec["x_local_cpu"], online

    b = REAL_BATCHES[step_idx]
    x_local = b["x_local"].to(DEVICE, non_blocking=True)
    x_context = b["x_context"].to(DEVICE, non_blocking=True)
    with torch.no_grad():
        online = extractor.extract_local_context(
            x_local,
            x_context,
            strategy="concat",
        )
    return b["path"], b["geometry"], b["x_local"], online


ALL_REPORTS = {}
LOG_ROWS = []

for candidate_id in CANDIDATES:
    print("\n" + "=" * 80)
    print("CANDIDATE:", candidate_id)
    print("=" * 80)

    # Re-seed để initialization downstream reproducible giữa candidates.
    seed_everything(SEED)

    model = Day05RepresentationSmokeModel(
        candidate_id=candidate_id,
        in_channels=int(extractor.out_channels),
    ).to(DEVICE)
    model.train()

    # MeanFusion/Aligner/Selector parameter-free.
    assert len(list(model.fusion.parameters())) == 0
    assert len(list(model.aligner.parameters())) == 0
    assert len(list(model.selector.parameters())) == 0

    # Optimizer SMOKE-ONLY. DINO không nằm trong optimizer.
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=SMOKE_LR,
        weight_decay=SMOKE_WEIGHT_DECAY,
    )

    before_all = snapshot_parameters(model)
    source_usage_steps = []
    step_logs = []
    last_logits = None
    last_x_local_cpu = None
    last_path = None

    for step in range(STEPS_PER_CANDIDATE):
        image_path, geometry, x_local_cpu, online = get_online_batch(step)

        # DINO frozen outputs are data tensors.
        assert all(not x.requires_grad for x in online.values())
        for p in BACKBONE_PARAMS:
            assert p.grad is None

        optimizer.zero_grad(set_to_none=True)

        logits, trace = model(
            online,
            geometry,
            output_size=(LOCAL_SIZE, LOCAL_SIZE),
            retain_source_grads=True,
        )

        assert logits.shape == (BATCH_SIZE, 1, LOCAL_SIZE, LOCAL_SIZE)
        assert torch.isfinite(logits).all()

        # MVTec AD 2 train là normal-only -> zero target là nhãn thật của train.
        # Dice term = differentiable zero cho normal-only batch; BCE vẫn tạo gradient.
        target = torch.zeros_like(logits)
        loss_out = criterion(logits, target)
        loss = loss_out["loss"]

        assert loss.ndim == 0
        assert torch.isfinite(loss)

        loss.backward()

        # ----------------------------------------------------
        # Source-usage audit: trực tiếp trên tensor trước MeanFusion.
        # ----------------------------------------------------
        usage = {}
        for source_key, tensor in trace["projected"].items():
            g = tensor.grad
            if g is None:
                raise AssertionError(
                    f"{candidate_id} step={step}: {source_key} bị disconnected."
                )
            if not torch.isfinite(g).all():
                raise AssertionError(
                    f"{candidate_id} step={step}: {source_key} gradient NaN/Inf."
                )
            grad_l1 = float(g.detach().abs().sum().item())
            if grad_l1 <= 0.0:
                raise AssertionError(
                    f"{candidate_id} step={step}: {source_key} gradient = 0; "
                    "source có thể không thực sự đóng góp vào graph."
                )
            usage[source_key] = {
                "grad_present": True,
                "finite": True,
                "grad_l1": grad_l1,
            }

        if candidate_id == "R2":
            expected_r2 = {
                "local_b4", "local_b8", "local_b12",
                "context_b4", "context_b8", "context_b12",
            }
            assert set(usage) == expected_r2
            assert len(usage) == 6

        grad_report = {
            "adapter": grad_stats_for_module(model.adapters),
            "projection": grad_stats_for_module(model.projection),
            "decoder": grad_stats_for_module(model.decoder),
        }
        for name, report in grad_report.items():
            if not report["finite"]:
                raise AssertionError(f"{candidate_id}:{name} gradient NaN/Inf")
            if report["with_grad"] == 0:
                raise AssertionError(f"{candidate_id}:{name} không có gradient")

        # MeanFusion parameter-free: source tensor gradients là gate đúng.
        optimizer.step()

        # Frozen DINO phải vẫn không có grad sau downstream backward/step.
        assert all(not p.requires_grad for p in BACKBONE_PARAMS)
        assert all(p.grad is None for p in BACKBONE_PARAMS)
        assert extractor.backbone.training is False

        row = {
            "candidate": candidate_id,
            "step": step + 1,
            "image": str(image_path),
            "loss": float(loss.detach().cpu()),
            "bce": float(loss_out["bce"].detach().cpu()),
            "dice": float(loss_out["dice"].detach().cpu()),
            "num_sources": int(model.selector.num_sources),
            "sources": list(model.selector.source_keys),
            "logits_min": float(logits.detach().amin().cpu()),
            "logits_max": float(logits.detach().amax().cpu()),
            "adapter_grad_l1": grad_report["adapter"]["grad_l1"],
            "projection_grad_l1": grad_report["projection"]["grad_l1"],
            "decoder_grad_l1": grad_report["decoder"]["grad_l1"],
        }
        step_logs.append(row)
        LOG_ROWS.append(row)
        source_usage_steps.append(usage)

        print(
            f"step {step+1}/{STEPS_PER_CANDIDATE} | "
            f"loss={row['loss']:.6f} | "
            f"sources={row['num_sources']} | "
            f"adapter_grad={row['adapter_grad_l1']:.3e}"
        )

        last_logits = logits.detach()
        last_x_local_cpu = x_local_cpu.detach().cpu()
        last_path = image_path

    # --------------------------------------------------------
    # Weight-change gate AFTER >=2 steps.
    # gamma_init=0: first step may only change gamma; subsequent steps allow
    # internal Adapter weights to move.
    # --------------------------------------------------------
    changed = changed_parameter_names(before_all, model)

    active_blocks = set(model.active_blocks)
    active_adapter_changed = {}
    inactive_adapter_changed = {}
    for block in BLOCKS:
        prefix = f"adapters.b{block}."
        names = [n for n in changed if n.startswith(prefix)]
        if block in active_blocks:
            active_adapter_changed[f"b{block}"] = names
            assert names, f"{candidate_id}: active Adapter b{block} không đổi weight."
        else:
            inactive_adapter_changed[f"b{block}"] = names
            assert not names, f"{candidate_id}: inactive Adapter b{block} lại đổi weight."

    assert any(n.startswith("projection.") for n in changed), (
        f"{candidate_id}: Projection không đổi weight."
    )
    assert any(n.startswith("decoder.") for n in changed), (
        f"{candidate_id}: Decoder không đổi weight."
    )

    # --------------------------------------------------------
    # Save anomaly map
    # --------------------------------------------------------
    cand_dir = OUTPUT_ROOT / candidate_id
    cand_dir.mkdir(parents=True, exist_ok=True)

    score = torch.sigmoid(last_logits[0, 0]).cpu().numpy()
    rgb = denormalize_local(last_x_local_cpu[0]).permute(1, 2, 0).numpy()

    fig = plt.figure(figsize=(12, 4))
    ax1 = fig.add_subplot(1, 3, 1)
    ax1.imshow(rgb)
    ax1.set_title(f"{candidate_id} Local train image")
    ax1.axis("off")

    ax2 = fig.add_subplot(1, 3, 2)
    im = ax2.imshow(score)
    ax2.set_title("Anomaly score map")
    ax2.axis("off")
    fig.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)

    ax3 = fig.add_subplot(1, 3, 3)
    ax3.imshow(rgb)
    ax3.imshow(score, alpha=0.45)
    ax3.set_title("Overlay (QA only)")
    ax3.axis("off")

    fig.tight_layout()
    map_path = cand_dir / "sample_anomaly_map.png"
    fig.savefig(map_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    candidate_report = {
        "status": "PASS",
        "candidate": candidate_id,
        "mode": model.selector.mode,
        "sources": list(model.selector.source_keys),
        "num_sources": model.selector.num_sources,
        "adapter_smoke_only": {
            "r": SMOKE_ADAPTER_R,
            "d": SMOKE_ADAPTER_D,
            "day04_winner": False,
        },
        "steps": step_logs,
        "source_usage": source_usage_steps,
        "changed_parameters": changed,
        "active_adapter_changed": active_adapter_changed,
        "inactive_adapter_changed": inactive_adapter_changed,
        "sample_anomaly_map": str(map_path),
        "last_image": str(last_path),
    }

    report_path = cand_dir / "integration_report.json"
    report_path.write_text(
        json.dumps(candidate_report, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    ALL_REPORTS[candidate_id] = candidate_report

    print("PASS:", candidate_id)
    print("report:", report_path)
    print("map   :", map_path)

print("\nAll candidates completed.")


In [ ]:

# ============================================================
# 10. FINAL R2 SOURCE-USAGE HARD GATE + EXPORT LOGS
# ============================================================
import csv

# Hard scientific plumbing gate for R2.
r2 = ALL_REPORTS["R2"]
expected = [
    "local_b4", "local_b8", "local_b12",
    "context_b4", "context_b8", "context_b12",
]
assert r2["sources"] == expected
assert r2["num_sources"] == 6

for step_idx, usage in enumerate(r2["source_usage"], start=1):
    assert list(usage.keys()) == expected
    for key in expected:
        assert usage[key]["grad_present"] is True
        assert usage[key]["finite"] is True
        assert usage[key]["grad_l1"] > 0.0

# CSV integration log.
csv_path = OUTPUT_ROOT / "integration_log.csv"
with csv_path.open("w", newline="", encoding="utf-8") as f:
    fieldnames = [
        "candidate", "step", "image", "loss", "bce", "dice",
        "num_sources", "sources",
        "logits_min", "logits_max",
        "adapter_grad_l1", "projection_grad_l1", "decoder_grad_l1",
    ]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for row in LOG_ROWS:
        out = dict(row)
        out["sources"] = ",".join(out["sources"])
        writer.writerow(out)

source_usage_path = OUTPUT_ROOT / "source_usage_gradient_report.json"
source_usage_path.write_text(
    json.dumps(
        {
            "status": "PASS",
            "R2_expected_sources": expected,
            "R2_steps": r2["source_usage"],
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

summary = {
    "gate": "TV1-B real-data integration",
    "status": "PASS",
    "scientific_result": False,
    "reason": "framework smoke only; temporary adapter r,d",
    "category": CATEGORY,
    "real_train_images": [str(b["path"]) for b in REAL_BATCHES],
    "backbone": {
        "name": MODEL_NAME,
        "frozen": extractor.backbone_is_frozen(),
        "requires_grad_true": sum(p.requires_grad for p in BACKBONE_PARAMS),
        "grads_present": sum(p.grad is not None for p in BACKBONE_PARAMS),
    },
    "adapter": {
        "r": SMOKE_ADAPTER_R,
        "d": SMOKE_ADAPTER_D,
        "source": "temporary_smoke_not_day04_winner",
    },
    "candidate_status": {
        c: {
            "status": ALL_REPORTS[c]["status"],
            "num_sources": ALL_REPORTS[c]["num_sources"],
            "sources": ALL_REPORTS[c]["sources"],
        }
        for c in CANDIDATES
    },
    "artifacts": {
        "integration_log_csv": str(csv_path),
        "source_usage_gradient_report_json": str(source_usage_path),
        "maps": {
            c: ALL_REPORTS[c]["sample_anomaly_map"]
            for c in CANDIDATES
        },
    },
}

summary_path = OUTPUT_ROOT / "integration_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("=" * 80)
print("TV1-B REAL-DATA INTEGRATION: PASS")
print("=" * 80)
print("R0 sources:", ALL_REPORTS["R0"]["num_sources"])
print("R1 sources:", ALL_REPORTS["R1"]["num_sources"])
print("R2 sources:", ALL_REPORTS["R2"]["num_sources"])
print("DINO trainable params:", summary["backbone"]["requires_grad_true"])
print("DINO grads present    :", summary["backbone"]["grads_present"])
print()
print("Integration log :", csv_path)
print("R2 usage report :", source_usage_path)
print("Summary         :", summary_path)
print("Output root     :", OUTPUT_ROOT)
print()
print("NOTE: PASS ở đây chỉ chứng minh framework/gradient/optimizer wiring đúng.")
print("      Không dùng loss/map của smoke run để xếp hạng R0/R1/R2.")


In [ ]:

# ============================================================
# 11. VISUAL QA — hiển thị 3 anomaly maps
# ============================================================
from IPython.display import display

fig = plt.figure(figsize=(15, 5))
for idx, candidate_id in enumerate(CANDIDATES, start=1):
    p = Path(ALL_REPORTS[candidate_id]["sample_anomaly_map"])
    img = Image.open(p)
    ax = fig.add_subplot(1, 3, idx)
    ax.imshow(img)
    ax.set_title(candidate_id)
    ax.axis("off")
fig.tight_layout()
plt.show()



## Khi Day-04 đã có `(r,d)` winner

Chỉ thay:

```python
SMOKE_ADAPTER_R = <winner_r>
SMOKE_ADAPTER_D = <winner_d>
```

sau đó chạy lại từ cell **2 → 10**.

Đối với training/ablation Day 05 chính thức, không nên tiếp tục gọi hai giá trị này là `SMOKE_*`; hãy ghi chúng vào config locked của Day 05 và dùng **cùng một `(r,d)` cho R0/R1/R2**.

### Không được kết luận từ notebook này

- Không so sánh loss sau 3 step để nói R0/R1/R2 cái nào tốt hơn.
- Không dùng anomaly map smoke để báo cáo AU-PRO.
- Không xem `r=32,d=128` là candidate tối ưu.

Notebook chỉ trả lời: **pipeline chạy thật không, gradient có đúng không, optimizer có cập nhật đúng branch không, và R2 có thật sự dùng đủ 6 sources không.**
